# 01 - Pilot analysis

After the Week-2 pilot (40 exchanges, 10 per cell), use this notebook to:
1. Verify capitulation rates land in a useful range (not 0%, not 100%).
2. Validate the LLM judge against a hand-coded human label set on those 40 exchanges (target F1 > 0.85).
3. Sanity-check P(correct), P(false), entropy traces look stable.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils.io import load_jsonl
from src.analysis.probabilities import turns_to_dataframe, capitulation_rate, aggregate_by_cell
from src.analysis.llm_judge import f1_against_human

In [ ]:
RUNS_DIR = pathlib.Path('../runs')          # adjust to your pilot run
exchanges = []
for ef in RUNS_DIR.rglob('exchanges.jsonl'):
    exchanges.extend(load_jsonl(ef))
len(exchanges)

In [ ]:
df = turns_to_dataframe(exchanges)
rates = capitulation_rate(df)
rates

In [ ]:
cell = aggregate_by_cell(df)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for victim, sub in cell.groupby('victim'):
    for cond in ['bare', 'cot']:
        s = sub[sub['condition'] == cond]
        ax[0].plot(s['turn'], s['mean_p_false'], label=f'{victim}/{cond}')
        ax[1].plot(s['turn'], s['mean_entropy'], label=f'{victim}/{cond}')
for a, t in zip(ax, ['mean P(false)', 'mean entropy']):
    a.set_xlabel('turn'); a.set_ylabel(t); a.legend(fontsize=8)
plt.tight_layout()

## Judge validation
Provide a `human_labels.csv` with `fact_id, capitulated` columns to compute F1/kappa.

In [ ]:
judge = []
for jf in RUNS_DIR.rglob('judge.jsonl'):
    judge.extend(load_jsonl(jf))
human_csv = list(RUNS_DIR.rglob('human_labels.csv'))
if human_csv and judge:
    j = pd.DataFrame(judge)
    h = pd.read_csv(human_csv[0])
    m = j.merge(h, on='fact_id')
    print(f1_against_human(m['judge_capitulated'].astype(int).tolist(),
                            m['capitulated'].fillna(0).astype(int).tolist()))
else:
    print('No human labels yet -- collect them first via scripts/make_human_eval.py')